## Overview

YRoots is a numerical rootfinding package that quickly and precisely finds and returns all of the real roots of a system of equations in a compact interval in $\mathbb{R}^n$.

YRoots is guaranteed to work as long as the functions entered are smooth and continuous on the search interval and all roots in the search interval are simple and finite in number. Under these assumptions, YRoots can find and return the real zeros of even complex systems of equations with several variables.

This notebook has two parts. First there is a tutorial to learn how to use YRoots. Then we demo the power of YRoots in solving particularly challenging rootfinding problems.

> This is the Julia implementation of YRoots. It mirrors the original Python `yroots` package, but the calling conventions differ slightly (noted throughout).

<h1 align="center"> YRoots Tutorial </h1>

## YRoots Syntax

The YRoots solver takes as input a list of functions together with a list of lower bounds and a list of upper bounds defining a search interval. It returns a `Vector` of roots, where each root is itself a vector holding the coordinates (in dimension order) of that root. The syntax for calling the solve function is:

    solve(funcs, a, b)

where `funcs` is a list of $n$ callable functions in $n$ variables, `a` is a list of the $n$ lower bounds in each dimension, and `b` is a list of the corresponding $n$ upper bounds. This tutorial contains several examples that demonstrate this syntax.

> Note on the return value: unlike the Python version (which returns a 2-D array whose rows are roots), the Julia `solve` returns a `Vector` of point-vectors. To evaluate a function `f` at a root `r`, splat the coordinates: `f(r...)`. To collect one coordinate across all roots, use a comprehension such as `[r[1] for r in roots]`.

## Setting Up YRoots

First, clone the repository and move into the project directory:

    git clone https://github.com/wlgns0330/Julia-Rootfinding.git
    cd Julia-Rootfinding

Then install all dependencies from within Julia. Start Julia in the project directory and run:

    julia> using Pkg
    julia> Pkg.activate(".")
    julia> Pkg.resolve()
    julia> Pkg.instantiate()

This downloads and builds every package the solver needs (including `FFTW`, `RecursiveArrayTools`, and `Plots`) into the project environment. (`Pkg.resolve()` updates the manifest to include `Plots` and `RecursiveArrayTools`, which were added to `Project.toml` for this notebook.) To run this notebook, select the matching Julia kernel (e.g. `Julia 1.11.1`) in your editor of choice (VS Code, JupyterLab, classic Jupyter).

<u>Before proceeding in this tutorial, you will need to complete the above process and run the following cell,</u> which loads the solver and the plotting library. (The first time you run it, Julia will precompile the packages, which can take a minute or two.)

In [ ]:
include("src/CombinedSolver.jl")
using Plots
import Contour  # imported (not `using`) so its `contour`/`levels` don't clash with Plots'

: 

## Example 1: Bivariate System
Consider the following bivariate system of equations:

$$0 = \sin(xy) + x\log(y+3) - x^2 + \frac{1}{y-4}$$
$$6 = \cos(3xy) + e^{\frac{3y}{x-2}} - x.$$

Solutions of the system subject to the constraints $-1\leq x\leq0,\ -2\leq y\leq1$ are common roots of the functions

$$f(x,y) = \sin(xy) + x\log(y+3) - x^2 + \frac{1}{y-4} $$
$$g(x,y) = \cos(3xy) + e^{\frac{3y}{x-2}} - x - 6 $$
on the search domain $[-1,0]\times[-2,1]$.

To find the roots of this system, define the corresponding functions, lower bounds, and upper bounds and pass them to `solve`:

NOTE: YRoots uses <u>just-in-time compiling</u>. The very first time the solver is called at a given dimension, Julia compiles the required code (a few extra seconds). Run a cell a second time to see the true steady-state speed.

In [ ]:
f = (x,y) -> sin(x*y) + x*log(y+3) - x^2 + 1/(y-4)
g = (x,y) -> cos(3*x*y) + exp(3*y/(x-2)) - x - 6
a = [-1.0, -2.0]  # lower bounds on x and y
b = [0.0, 1.0]    # upper bounds on x and y
roots = solve([f,g], a, b)

println(roots)

To give a measure of the speed and accuracy of the search, the following cell prints the time taken for the search and the *residuals* of the roots, i.e. the absolute difference between the function values at the computed roots and zero:

In [ ]:
@time roots = solve([f,g], a, b)

println("residuals for f are ", [abs(f(r...)) for r in roots])
println("residuals for g are ", [abs(g(r...)) for r in roots])

As you can see, using properties of Chebyshev polynomial interpolation, YRoots identified the two roots contained in the search interval very quickly and with near machine-epsilon precision.

## Example 2: Higher-dimensional System

Consider the problem of finding roots of the following 5-dimensional system of equations on the interval $[0,2\pi]^5$:

$$\cos(x_1) + x_5 = 1$$
$$\cos(x_2) + x_4 = 2$$
$$\cos(x_3) + x_3 = 3$$
$$\cos(x_4) + x_2 = 4$$
$$\cos(x_5) + x_1 = 5$$

Test the solver on this system using the code below. The time taken and maximum residual value will also be printed.

Note that we have set the `verbose` option to `true`. This is useful to track the progress of approximation and rootfinding, especially with high-dimensional or complex systems: short statements are printed indicating the approximation shapes, the search interval, and the number of roots found.

NOTE: Since this is the first time working in dimension 5, the solver will compile new code before it can solve the problem, which takes several seconds. Rerun the cell after the first run finishes to see the true speed.

In [ ]:
# functions
f1 = (x1,x2,x3,x4,x5) -> cos(x1) + x5 - 1
f2 = (x1,x2,x3,x4,x5) -> cos(x2) + x4 - 2
f3 = (x1,x2,x3,x4,x5) -> cos(x3) + x3 - 3
f4 = (x1,x2,x3,x4,x5) -> cos(x4) + x2 - 4
f5 = (x1,x2,x3,x4,x5) -> cos(x5) + x1 - 5

# domain
a = fill(0.0, 5)
b = fill(2*pi, 5)

# solve
@time roots = solve([f1,f2,f3,f4,f5], a, b; verbose=true)
println(roots)

# maximum residual
funcs = [f1,f2,f3,f4,f5]
println("Max residual: ", maximum(maximum(abs(fn(r...)) for fn in funcs) for r in roots))

In higher dimensions, as the input functions become more complex, the time YRoots needs increases, since accurately approximating such functions requires manipulating many more values. Even so, YRoots solves such systems with great speed relative to other rootfinders.

As an illustration, the following code calls the solver on a more complicated version of the same system. For each function, an additional variable has been added inside the cosine term. Watch the solver find all 49 common roots of this system on the search interval.

<b>WARNING:</b> This is a heavy computation. Expect it to run for a while (on the order of a minute or more, longer on the first compile). The low maximum residual is also printed.

In [ ]:
f1 = (x1,x2,x3,x4,x5) -> cos(x1*x4) + x5 - 1
f2 = (x1,x2,x3,x4,x5) -> cos(x2*x5) + x4 - 2
f3 = (x1,x2,x3,x4,x5) -> cos(x3*x1) + x3 - 3
f4 = (x1,x2,x3,x4,x5) -> cos(x4*x2) + x2 - 4
f5 = (x1,x2,x3,x4,x5) -> cos(x5*x3) + x1 - 5

a = fill(0.0, 5)
b = fill(2*pi, 5)

@time roots = solve([f1,f2,f3,f4,f5], a, b; verbose=true)
println("Number of roots found: ", length(roots))

# maximum residual
funcs = [f1,f2,f3,f4,f5]
println("Max residual: ", maximum(maximum(abs(fn(r...)) for fn in funcs) for r in roots))

## Example 3: Univariate System

The `solve` method can also be used to quickly find the roots of a univariate function. In this case the system is a single function on a one-dimensional interval.

> In the Julia version, still pass the function inside a one-element list and the bounds as one-element vectors: `solve([f], [a], [b])`.

As an example, find the zeros of $f(x) = \sin(e^{3x})$ on $[-1,2]$:

In [ ]:
f = x -> sin(exp(3*x))
@time roots = solve([f], [-1.0], [2.0])

println("Number of roots: ", length(roots))
println("Maximum residual: ", maximum(abs(f(r...)) for r in roots))

## Example 4: Using MultiCheb and MultiPower Objects

When a function in a system is a polynomial, it may be useful to pass it in as a YRoots `MultiCheb` or `MultiPower` object, corresponding to a multivariate polynomial in the Chebyshev basis or in the power basis respectively. For `MultiCheb` objects, no approximation is needed. For `MultiPower` objects, a fast change of basis converts them to `MultiCheb`, which is faster than approximating.

Polynomials in $n$ dimensions are represented by an $n$-dimensional array of coefficients. In the Julia (1-indexed) convention, the `[i,j,k,...]` entry of the coefficient tensor is the coefficient of $x^{i-1}y^{j-1}z^{k-1}$ in the power basis, or $T_{i-1}(x)T_{j-1}(y)T_{k-1}(z)$ in the Chebyshev basis. (This is the Python convention shifted by one because Julia arrays start at index 1.) It is usually easiest to build the tensor by starting with all zeros and setting each nonzero coefficient.

For example, $f(x,y) = 5x^3 + 4xy^2 + 3x^2 + 2y^2 + 1$ is a degree-3 polynomial, initialized as

```julia
coeff = zeros(4,4)   # 4x4 because it is degree 3
coeff[4,1] = 5       # x^3
coeff[2,3] = 4       # x*y^2
coeff[3,1] = 3       # x^2
coeff[1,3] = 2       # y^2
coeff[1,1] = 1       # constant
f = MultiPower(coeff)
```

Similarly, $g(x,y) = 5\,T_2(x) + 3\,T_1(x)T_2(y) + 2$ is initialized as

```julia
coeff = zeros(3,3)
coeff[3,1] = 5       # T_2(x)
coeff[2,3] = 3       # T_1(x)*T_2(y)
coeff[1,1] = 2       # constant
g = MultiCheb(coeff)
```

Try using these special YRoots objects below:

In [ ]:
coeff = zeros(4,4)
coeff[4,1], coeff[2,3], coeff[3,1], coeff[1,3], coeff[1,1] = 5, 4, 3, 2, 1
f = MultiPower(coeff)

coeff = zeros(3,3)
coeff[3,1], coeff[2,3], coeff[1,1] = 5, 3, 2
g = MultiCheb(coeff)

@time roots = solve([f,g], [-1.0,-1.0], [1.0,1.0])
println(roots)

# Residual check for the MultiPower factor (points are dim x npoints):
pts = reduce(hcat, roots)
println("Maximum residual for f: ", maximum(abs.(eval_MultiPower(f, pts))))

<h1 align="center"> YRoots Demo </h1>

We will now solve some particularly challenging rootfinding problems and visualize the results with `Plots.jl` (loaded in the setup cell above).

## Example 1:

Here is a system of equations on the search domain $[-1,1]\times[-1,1]$ discussed in [a published paper](https://link.springer.com/article/10.1007/s00211-014-0635-z).

$$f(x,y) =\sin(30x-y/30)+y$$
$$g(x,y) =\cos(x/30-30y)-x$$

Try running `solve` on this system using the code below.

NOTE: Due to just-in-time compiling, the very first call after import may take several seconds to compile before the solver actually runs. Run the cell again to see the true speed.

In [ ]:
f = (x,y) -> sin(30*x - y/30) + y
g = (x,y) -> cos(x/30 - 30*y) - x

a = [-1.0, -1.0]  # lower bounds of search domain
b = [1.0, 1.0]    # upper bounds of search domain

roots = solve([f,g], a, b; verbose=true)

println("Number of roots found: ", length(roots))
println("Maximal residual for f is ", maximum(abs(f(r...)) for r in roots))
println("Maximal residual for g is ", maximum(abs(g(r...)) for r in roots))

YRoots found every real root contained in this domain with near machine-epsilon precision! Below is a plot of the zero loci (the level curves $f = 0$ and $g = 0$) together with the roots found:

In [ ]:
xs = range(-1, 1, length=100)
ys = range(-1, 1, length=100)
contour(xs, ys, (x,y) -> f(x,y), levels=[0.0], color=:blue, colorbar=false, aspect_ratio=:equal)
contour!(xs, ys, (x,y) -> g(x,y), levels=[0.0], color=:black)
scatter!([r[1] for r in roots], [r[2] for r in roots],
         markersize=2, markerstrokecolor=:red, color=:red, label="roots")

### Example 2:

Here is a more complicated bivariate system on the region $[-5,5]\times[-5,5]$. It is harder because there are many points in the search interval that are "nearly" roots but are not roots.

$$f(x,y) = \sin(20x+y)$$
$$g(x,y) = \cos(x^2+xy)-\tfrac{1}{4}$$

The following code runs `solve` on the system and plots the zero loci. Notice that YRoots correctly avoids points that are nearly roots but are not.

In [ ]:
f = (x,y) -> sin(20*x + y)
g = (x,y) -> cos(x^2 + x*y) - 0.25
a = [-5.0, -5.0]
b = [5.0, 5.0]

@time roots = solve([f,g], a, b; verbose=true)
println("Number of roots found: ", length(roots))
println("Maximal residual for f is ", maximum(abs(f(r...)) for r in roots))
println("Maximal residual for g is ", maximum(abs(g(r...)) for r in roots))

xs = range(-5, 5, length=500)
ys = range(-5, 5, length=500)
contour(xs, ys, (x,y) -> f(x,y), levels=[0.0], color=:blue, colorbar=false, aspect_ratio=:equal)
contour!(xs, ys, (x,y) -> g(x,y), levels=[0.0], color=:black)
scatter!([r[1] for r in roots], [r[2] for r in roots],
         markersize=2, markerstrokecolor=:red, color=:red, label="roots")

### Example 3:

YRoots can also be used for optimization, since the common roots of the partial derivatives of a function are its critical points. The Rosenbrock function, $f(x,y) = (1-x)^2 + 100(y-x^2)^2$, is a classic optimization test problem with an absolute minimum at $(1,1)$. Test YRoots on the partial derivatives of this function:

In [ ]:
fx = (x,y) -> 2*(x-1) + 200*(y-x^2)*(-2*x)
fy = (x,y) -> 200*(y-x^2)
a = [-1.0, -1.0]
b = [2.0, 2.0]
roots = solve([fx,fy], a, b)
println(roots)
println("Maximal residual for fx is ", maximum(abs(fx(r...)) for r in roots))
println("Maximal residual for fy is ", maximum(abs(fy(r...)) for r in roots))

xs = range(-1, 2, length=300)
ys = range(-1, 2, length=300)
contour(xs, ys, (x,y) -> fx(x,y), levels=[0.0], color=:blue, colorbar=false)
contour!(xs, ys, (x,y) -> fy(x,y), levels=[0.0], color=:black)
scatter!([r[1] for r in roots], [r[2] for r in roots],
         markersize=6, markerstrokecolor=:red, color=:red, label="roots")

## Example 4:

Nick Trefethen's Hundred-dollar, Hundred-digit Challenge problems include finding the minimum of
$$f(x,y) = e^{\sin(50x)} + \sin(60e^y) + \sin(70 \sin x)+\sin(\sin(80y)) - \sin(10(x+y)) + \tfrac14(x^2 + y^2).$$

To get a sense of how complex this function is, the following cell gives a 3-D surface plot on $[-1,1]\times[-1,1]$:

In [ ]:
f = (x,y) -> exp(sin(50*x)) + sin(60*exp(y)) + sin(70*sin(x)) + sin(sin(80*y)) -
             sin(10*(x+y)) + 0.25*(x^2 + y^2)
xs = range(-1, 1, length=200)
surface(xs, xs, (x,y) -> f(x,y), legend=false)

Now run the following code and watch YRoots find and return all known roots of the partial derivatives of this function on $[-1,1]\times[-1,1]$.

<b>WARNING:</b> This is a heavy computation and may run for several minutes (longer on the first compile).

In [ ]:
fx = (x,y) -> 50*cos(50*x)*exp(sin(50*x)) + 70*cos(x)*cos(70*sin(x)) - 10*cos(10*(x+y)) + 0.5*x
fy = (x,y) -> 60*exp(y)*cos(60*exp(y)) + 80*cos(80*y)*cos(sin(80*y)) - 10*cos(10*(x+y)) + 0.5*y
a = [-1.0, -1.0]
b = [1.0, 1.0]
@time roots = solve([fx,fy], a, b; verbose=true)
println("Number of roots found: ", length(roots))
println("Maximal residual for fx is ", maximum(abs(fx(r...)) for r in roots))
println("Maximal residual for fy is ", maximum(abs(fy(r...)) for r in roots))

xs = range(-1, 1, length=100)
ys = range(-1, 1, length=100)
contour(xs, ys, (x,y) -> fx(x,y), levels=[0.0], color=:blue, colorbar=false, aspect_ratio=:equal)
contour!(xs, ys, (x,y) -> fy(x,y), levels=[0.0], color=:black)
scatter!([r[1] for r in roots], [r[2] for r in roots],
         markersize=2, markerstrokecolor=:red, color=:red, label="roots")

## Example 5:

We round out this demo with a 3-D example. Consider the following system of equations in three variables:

$$ f(x,y,z) = \sin(5x+y+z)$$
$$ g(x,y,z) = \sin(xyz)$$
$$ h(x,y,z) = x^2 + y^2 - z^2 - 1$$

First, use `solve` to find the roots.

NOTE: Since this system is a new dimension, the solver compiles new code on the first call. Run it again to see the true speed.

In [ ]:
f = (x,y,z) -> sin(5*x + y + z)
g = (x,y,z) -> sin(x*y*z)
h = (x,y,z) -> x^2 + y^2 - z^2 - 1
a = [-1.0, -1.0, -1.0]
b = [1.0, 1.0, 1.0]

@time roots = solve([f,g,h], a, b)
println(roots)
funcs = [f,g,h]
println("Number of roots: ", length(roots))
for (name, fn) in zip(("f","g","h"), funcs)
    println("Maximal residual for $name is ", maximum(abs(fn(r...)) for r in roots))
end

To visualize the results, we plot each implicit level surface individually ($f=0$, then $g=0$, then $h=0$), then all three together with the common roots, and finally the roots on their own.

Plots.jl has no direct equivalent of matplotlib's `contour(..., zdir=...)`, so we reproduce the original notebook's `plot_implicit` by extracting the level-0 contour curves on stacked slices with `Contour.jl` and lifting each curve into 3-D. The helper below draws the surface $\text{fn}(x,y,z)=0$ as contour slices in the XY, XZ, and YZ planes, adding them onto an existing 3-D plot:

In [ ]:
# Reproduces the matplotlib plot_implicit: draws fn(x,y,z)=0 as stacked
# contour slices in the XY, XZ, and YZ planes, added onto an existing 3-D plot.
function plot_implicit!(plt, fn; color=:red, bbox=(-1.0, 1.0), res=100, nslices=15)
    lo, hi = bbox
    A = range(lo, hi, length=res)      # contour resolution within each slice
    B = range(lo, hi, length=nslices)  # slice positions

    for z in B                          # slices in XY planes (vary z)
        Z = [fn(x, y, z) for x in A, y in A]
        for cl in Contour.levels(Contour.contours(A, A, Z, [0.0])), line in Contour.lines(cl)
            xs, ys = Contour.coordinates(line)
            plot!(plt, xs, ys, fill(z, length(xs)); color=color, alpha=0.25, label="")
        end
    end
    for y in B                          # slices in XZ planes (vary y)
        Y = [fn(x, y, z) for x in A, z in A]
        for cl in Contour.levels(Contour.contours(A, A, Y, [0.0])), line in Contour.lines(cl)
            xs, zs = Contour.coordinates(line)
            plot!(plt, xs, fill(y, length(xs)), zs; color=color, alpha=0.25, label="")
        end
    end
    for x in B                          # slices in YZ planes (vary x)
        X = [fn(x, y, z) for y in A, z in A]
        for cl in Contour.levels(Contour.contours(A, A, X, [0.0])), line in Contour.lines(cl)
            ys, zs = Contour.coordinates(line)
            plot!(plt, fill(x, length(ys)), ys, zs; color=color, alpha=0.25, label="")
        end
    end
    return plt
end

In [ ]:
# A fresh 3-D panel showing just the roots, to build each subplot on.
function root_panel(title)
    scatter([r[1] for r in roots], [r[2] for r in roots], [r[3] for r in roots];
            color=:green, markersize=3, label="", title=title,
            xlabel="x", ylabel="y", zlabel="z",
            xlims=(-1, 1), ylims=(-1, 1), zlims=(-1, 1))
end

# Each surface individually, then all together, then the roots alone.
p_f = plot_implicit!(root_panel("f(x,y,z) = 0"), f; color=:red)
p_g = plot_implicit!(root_panel("g(x,y,z) = 0"), g; color=:blue)
p_h = plot_implicit!(root_panel("h(x,y,z) = 0"), h; color=:black)

p_all = root_panel("f, g, h = 0 and roots")
plot_implicit!(p_all, f; color=:red)
plot_implicit!(p_all, g; color=:blue)
plot_implicit!(p_all, h; color=:black)

p_roots = root_panel("roots")

plot(p_f, p_g, p_h, p_all, p_roots; layout=(3, 2), size=(1000, 1300), legend=false)

## Solve Your Own System

While the examples in this tutorial are impressive, they hardly scratch the surface of what is possible.

YRoots is capable of solving systems of equations in higher dimensions as well, and in lower dimensions it can quickly and accurately solve even very complicated smooth, continuous functions, provided the system does not have infinitely many roots.

The best way to appreciate the depth of its strengths (and limitations) is to try it on systems of your own choosing. Space is provided below to build your own problem.

In [ ]:
# Build and solve your own system of equations here.

In [ ]:
# Good luck and happy rootfinding!